# Unidade III — Pré-processamento de Dados

## Transformação, codificação e discretização

**Carga estimada:** 3 horas  
**Pré-requisitos:** pandas, medidas de posição e fluxo supervisionado básico.

> **Pergunta norteadora:** como representar atributos em uma forma adequada ao algoritmo sem usar informação que estaria indisponível no momento da previsão?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- diferenciar normalização min–max e padronização;
- transformar distribuições assimétricas e codificar categorias;
- comparar discretização por largura e por frequência;
- usar pipelines ajustados somente com o conjunto de treino.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Escala e forma

A normalização min–max leva um valor $x$ ao intervalo desejado:

$$x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}.$$

A padronização usa média $\mu$ e desvio-padrão $\sigma$:

$$z = \frac{x-\mu}{\sigma}.$$

Min–max é sensível aos extremos e pode produzir valores fora do intervalo em novos dados. Padronização não torna a distribuição normal; apenas muda centro e escala. Para valores não negativos e assimétricos, `log1p` comprime a cauda e aceita zero.


In [2]:
n = 500
dados = pd.DataFrame({
    "idade": rng.integers(18, 76, n).astype(float),
    "renda": rng.lognormal(mean=8.2, sigma=0.7, size=n),
    "cidade": rng.choice(["Recife", "Olinda", "Paulista"], n, p=[0.55, 0.25, 0.20]),
    "chamados": rng.poisson(1.8, n),
})
dados.loc[rng.choice(n, 24, replace=False), "idade"] = np.nan
logito = -2.0 + 0.45 * dados["chamados"] - 0.00012 * dados["renda"]
dados["cancelou"] = rng.binomial(1, 1 / (1 + np.exp(-logito)))
dados.head()


,idade,renda,cidade,chamados,cancelou
0,23.0,3408.461216,Olinda,1,0
1,62.0,1063.786643,Olinda,1,0
2,55.0,1303.839078,Paulista,1,0
3,43.0,16162.887183,Recife,0,0
4,43.0,1478.530833,Paulista,3,0


In [3]:
resumo_renda = pd.DataFrame({
    "original": dados["renda"],
    "log1p": np.log1p(dados["renda"]),
}).agg(["mean", "median", "std", "skew"]).round(2)
resumo_renda


,original,log1p
mean,4533.22,8.17
median,3601.40,8.19
std,3535.01,0.71
skew,2.26,-0.06


## Categorias e discretização

Codificar cidades como 0, 1 e 2 introduziria uma ordem artificial. A codificação *one-hot* cria um indicador por categoria. Categorias novas exigem uma política explícita; abaixo, `handle_unknown="ignore"` produz zeros nos indicadores conhecidos.

Discretizar converte um atributo numérico em intervalos. Larguras iguais facilitam interpretar a escala, mas podem gerar intervalos vazios; frequências aproximadamente iguais equilibram contagens, mas os limites dependem da amostra e valores iguais podem dificultar a divisão. Há perda de informação dentro de cada intervalo.


In [4]:
renda = dados[["renda"]]
por_largura = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
por_frequencia = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
comparacao_bins = pd.DataFrame({
    "largura_igual": por_largura.fit_transform(renda).ravel().astype(int),
    "frequencia_igual": por_frequencia.fit_transform(renda).ravel().astype(int),
})
comparacao_bins.apply(pd.Series.value_counts).fillna(0).astype(int)


,largura_igual,frequencia_igual
0,426,125
1,61,125
2,10,125
3,3,125


## Pipeline e prevenção de vazamento

Vazamento ocorre quando o treinamento recebe informação que não estaria legitimamente disponível na aplicação. Calcular mediana, média, desvio ou categorias antes da partição deixa o teste influenciar o preparo. A sequência correta é: separar primeiro; ajustar (`fit`) transformações no treino; apenas aplicar (`transform`) ao teste. Um `Pipeline` registra essa ordem e deve envolver também o modelo em uma avaliação futura.


In [5]:
X = dados.drop(columns="cancelou")
y = dados["cancelou"]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numericas = ["idade", "renda", "chamados"]
categoricas = ["cidade"]
pipeline_numerico = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler()),
])
preprocessador = ColumnTransformer([
    ("num", pipeline_numerico, numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
])
X_treino_pronto = preprocessador.fit_transform(X_treino)
X_teste_pronto = preprocessador.transform(X_teste)
pd.Series({"linhas_treino": len(X_treino_pronto), "linhas_teste": len(X_teste_pronto), "atributos_saida": X_treino_pronto.shape[1]})


linhas_treino      375
linhas_teste       125
atributos_saida      6
dtype: int64

A mediana armazenada pelo imputador, as estatísticas do escalonador e as categorias do codificador vieram exclusivamente das 375 linhas de treino. A variável-alvo não foi usada pelas transformações. Discretização supervisionada, diferentemente, pode usar o alvo para escolher cortes, mas precisa ficar dentro do mesmo fluxo de treino e validação.

> **U03-NB02-V01 — Verifique seu entendimento:** por que chamar `fit_transform` separadamente no treino e no teste é incorreto, mesmo sem usar explicitamente a variável-alvo?

> **U03-NB02-E01 — Exercício:** recupere do pipeline a mediana aprendida para `idade` e compare-a à mediana do teste. Explique qual delas deve ser usada para transformar o teste e por quê.


## Síntese

- Escalonamento altera representação, não corrige qualidade nem garante normalidade.
- Codificação deve respeitar o significado do atributo.
- Discretização simplifica, mas perde resolução e aprende limites da amostra.
- Todo parâmetro de pré-processamento deve ser aprendido sem acesso ao teste.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seções 2.5–2.6.
- SCIKIT-LEARN DEVELOPERS. *User Guide*: preprocessing data e pipelines. Versão 1.x.
